# Kinetic Parameter Estimation

A kinetic parameter estimation problem originally described by [[1](#references)] is used to demonstrate the use of EOptInterface to formulate an `InfiniteOpt.InfiniteModel` [[2](#references)] from an ODE `ModelingToolkit.System` [[3](#references)]. The reacting system of interest is represented by the following initial value problem:
$$
\begin{align*}
    &\frac{dx_A}{dt} = k_1 x_Z x_Y - c_{\text{O}_{2}} (k_{2f} + k_{3f}) x_A + \frac{k_{2f}}{K_2} x_D + \frac{k_{3f}}{K_3} x_B - k_5 x_A^2 \\
    &\frac{dx_B}{dt} = c_{\text{O}_{2}} k_{3f} x_A - \left(\frac{k_{3f}}{K_3} + k_4\right) x_B \\
    &\frac{dx_D}{dt} = c_{\text{O}_{2}} k_{2f} x_A - \frac{k_{2f}}{K_2} x_D \\
    &\frac{dx_Y}{dt} = -k_1 x_Z x_Y \\
    &\frac{dx_Z}{dt} = -k_1 x_Z x_Y \\
    &x_A(0) = x_B(0) = x_D(0) = 0, \quad x_Y(0) = 0.4, \quad x_Z(0) = 140,
\end{align*}
$$
where $x_j$ is the concentration of species $j ∈ \{A, B, D, Y, Z \}$ and $T = 273, K_2 = 46\exp(6500/T-18), K_3 = 2 K_2, k_1 = 53, k_{1s} = k_1 × 10^{-6}, k_5 = 1.2 × 10^{-3},$ and $c_{\text{O}_{2}} = 2 × 10^{-3}$ are known constants. The remaining uncertain reaction rate constants $\mathbf{p} = (k_{2f}, k_{3f}, k_4)$ lie within the interval $P = [10,1200] × [10,1200] × [0.001,40]$. 

In [1]:
using CSV
using DataFrames
using EOptInterface
using InfiniteOpt
using Ipopt
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as D

This ODE system is modeled in `ModelingToolkit` with the unknown parameters and initial conditions defined as shown below.

In [2]:
@mtkmodel KineticParameterEstimation begin
    @parameters begin
        # Known parameters
        T = 273
        K_2 = 46*exp(6500/T-18)
        K_3 = 2*K_2
        k_1 = 53
        k_1s = k_1*1e-6
        k_5 = 1.2e-3
        c_O2 = 2e-3

        # Unknown parameters (free design variables)
        k_2f
        k_3f
        k_4
    end
    @variables begin
        # Initial conditions given for differential variables
        x_A(t) = 0.0
        x_B(t) = 0.0
        x_D(t) = 0.0
        x_Y(t) = 0.4
        x_Z(t) = 140.0
        
        I(t)
    end
    @equations begin
        D(x_A) ~ k_1*x_Z*x_Y - c_O2*(k_2f + k_3f)*x_A + k_2f/K_2*x_D + k_3f/K_3*x_B - k_5*x_A^2
        D(x_B) ~ c_O2*k_3f*x_A - (k_3f/K_3 + k_4)*x_B
        D(x_D) ~ c_O2*k_2f*x_A - k_2f/K_2*x_D
        D(x_Y) ~ -k_1s*x_Z*x_Y
        D(x_Z) ~ -k_1*x_Z*x_Y
        I ~ x_A + 2/21*x_B + 2/21*x_D
    end
end

@mtkcompile system = KineticParameterEstimation();

The objective is to determine the values of uncertain kinetic parameters that best fit this model against experimental intensity versus time data available in `kinetic_intensity_data.csv`, where the relationship between intensity and concentration is known to be $I^{\text{calc}} = x_A + \frac{2}{21} x_B + \frac{2}{21} x_D$ [[4](#references)]. The experimental data set $I^{\text{exp}}$ contains 201 measurements between $(0, 2)$ spaced linearly by $\Delta t = 0.01$. Based on this information, we define the integration time span to be $(0, 2)$ to include the initial condition, and use the same time step as between measurements, resulting in $N = 201$ discrete time points. The objective function is then defined as sum of squared error between the model-predicted and measured intensity:
$$
\begin{align}
    f(\mathbf{x}) = \sum_{i=2}^{N} \left(I^{\text{calc}}(\mathbf{x}_{i}) - I^{\text{exp}}_{i} \right)^2.
\end{align}
$$

In [11]:
data = CSV.read("kinetic_intensity_data.csv", DataFrame)
intensity(x_A, x_B, x_D) = x_A + 2/21*x_B + 2/21*x_D
tspan = [data.time[1], data.time[end]]
N = length(data.time);

We can then define the `InfiniteOpt.InfiniteModel` using an appropriate solver of choice. Given that this is a nonlinear system, we have chosen to use `Ipopt` [[5](#references)].

In [4]:
model = InfiniteModel(Ipopt.Optimizer);

We can use `InfiniteOpt.@infinite_parameter` to define our time span and number of supports, while also selecting our desired numerical method to evaluate derivatives.

In [12]:
@infinite_parameter(model, τ ∈ tspan, num_supports = N, derivative_method = OrthogonalCollocation(2));

We can use `EOptInterface.decision_vars` on the `ModelingToolkit.System` to retrieve the decision variables for the optimization problem.

In [6]:
decision_vars(system)

8-element Vector{SymbolicUtils.BasicSymbolicImpl.var"typeof(BasicSymbolicImpl)"{SymReal}}:
 x_Z(t)
 x_Y(t)
 x_D(t)
 x_B(t)
 x_A(t)
 k_2f
 k_3f
 k_4

Now that we know which variables appear in the system, we can add the decision variables to the `InfiniteOpt.InfiniteModel` with appropriate bounds.

In [13]:
V = length(unknowns(system))
@variable(model, -75.0 <= z[1:V] <= 150.0, Infinite(τ))
pL = [10.0, 10.0, 0.001]
pU = [1200.0, 1200.0, 40.0]
@variable(model, pL[i] <= p[i=1:3] <= pU[i]);

We then use `EOptInterface.register_odesystem` to register the ODE `ModelingToolkit.System` using explicit Euler and register the system as differential constraints.

In [ ]:
register_odesystem(model, system, τ)

We can add the objective function defined in Equation 1 to the `InfiniteOpt.InfiniteModel`.

In [16]:
@objective(model, Min, sum((intensity(z[5](i), z[4](i), z[3](i)) - data.intensity[findfirst(==(i), data.time)])^2 for i in supports(τ)));

Finally, we optimize the `InfiniteOpt.InfiniteModel` and retrieve the results.

In [17]:
optimize!(model)
println("Termination Status: $(termination_status(model))")
println("Primal Status: $(primal_status(model))")
println("Solve Time: $(round.(solve_time(model), digits=5))")
println("f^* = $(round(objective_value(model), digits=5))")
println("p* = $(round.(value.(p), digits=3))")


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.2.

Number of nonzeros in equality constraint Jacobian...:     9035
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:     3819

Total number of variables............................:     2013
                     variables with only lower bounds:        0
                variables with lower and upper bounds:     1008
                     variables with only upper bounds:        0
Total number of equality constraints.................:     2010
Total number of inequality c

### References

1. Taylor, J.W. Direct measurement and analysis of cyclohexadienyl oxidation. Ph.D. thesis, Massachusetts Institute of Technology. (2005). URL: http://hdl.handle.net/1721.1/33716
2. Pulsipher, J.L., Zhang, W., Hongisto, T.J., and Zavala, V.M. A unifying modeling abstraction for infinite-dimensional optimization. *Computers & Chemical Engineering.* 156, 107567 (2022). DOI: [10.1016/j.compchemeng.2021.107567](https://doi.org/10.1016/j.compchemeng.2021.107567)
3. Ma, Y., Gowda, S., Anantharaman, R., Laughman, C., Shah, V., and Rackauckas, C. ModelingToolkit: A Composable Graph Transformation System For Equation-Based Modeling. (2022). DOI: [10.48550/arXiv.2103.05244](https://doi.org/10.48550/arXiv.2103.05244)
3. Singer, A.B. Global dynamic optimization. Ph.D. thesis, Massachusetts Institute of Technology. (2004). URL: http://hdl.handle.net/1721.1/28662
4. Wächter, A. and Biegler, L.T. On the implementation of an interior-point filter line-search algorithm for large-scale nonlinear programming. *Mathematical Programming.* 106, 25-57 (2006). DOI: [10.1007/s10107-004-0559-y](https://doi.org/10.1007/s10107-004-0559-y)